In [1]:
import os
import gc
import re
import json
import math
import random
import warnings

import numpy as np
import polars as pl

from pathlib import Path
from tqdm.auto import tqdm
from rapidfuzz import fuzz

from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import precision_score, recall_score

import lightgbm as lgb

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Polars:", pl.__version__)
print("LightGBM:", lgb.__version__)

d:\business_entity_resolution\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Polars: 1.44.2
LightGBM: 4.7.0


In [4]:
DATA_ROOT = Path(".")

TRAIN_DIR = DATA_ROOT / "dataset" / "train"
TEST_DIR = DATA_ROOT / "dataset" / "test"

TRAIN_S1 = TRAIN_DIR / "train_source1.tsv"
TRAIN_S2 = TRAIN_DIR / "train_source2.tsv"
TRAIN_S3 = TRAIN_DIR / "train_source3.tsv"
GROUND_TRUTH = TRAIN_DIR / "train_ground_truth.tsv"

TEST_S1 = TEST_DIR / "test_source1.tsv"
TEST_S2 = TEST_DIR / "test_source2.tsv"
TEST_S3 = TEST_DIR / "test_source3.tsv"

OUTPUT_DIR = DATA_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Train directory:", TRAIN_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())

Train directory: D:\business_entity_resolution\dataset\train
Output directory: D:\business_entity_resolution\output


In [5]:
files = [
    TRAIN_S1,
    TRAIN_S2,
    TRAIN_S3,
    GROUND_TRUTH,
    TEST_S1,
    TEST_S2,
    TEST_S3,
]

for path in files:
    if path.exists():
        size_gb = path.stat().st_size / (1024 ** 3)
        print(f"{path.name:30s} {size_gb:.3f} GB")
    else:
        print(f"MISSING: {path}")

train_source1.tsv              0.196 GB
train_source2.tsv              0.456 GB
train_source3.tsv              0.469 GB
train_ground_truth.tsv         0.118 GB
test_source1.tsv               0.163 GB
test_source2.tsv               0.474 GB
test_source3.tsv               0.471 GB


In [6]:
for path in [TRAIN_S1, TRAIN_S2, TRAIN_S3]:
    print(f"\n{'=' * 60}")
    print(path.name)
    
    df = pl.read_csv(
        path,
        separator="\t",
        n_rows=5,
        infer_schema_length=1000,
    )
    
    print(df)
    print("\nSchema:")
    print(df.schema)


train_source1.tsv
shape: (5, 4)
┌──────────────┬─────────────────────────┬─────────────────────────────────┬─────────┐
│ entity_id    ┆ business_name           ┆ business_address                ┆ country │
│ ---          ┆ ---                     ┆ ---                             ┆ ---     │
│ str          ┆ str                     ┆ str                             ┆ str     │
╞══════════════╪═════════════════════════╪═════════════════════════════════╪═════════╡
│ S1-925783039 ┆ Orelee's Barbershop     ┆ 1795 Westchester Drive, High P… ┆ US      │
│ S1-773889195 ┆ Prime Money             ┆ 17560 Ellis Road, Tahlequah, O… ┆ US      │
│ S1-377745466 ┆ B+ Retail Inc           ┆ 1712 Montebello Avenue, Phoeni… ┆ US      │
│ S1-133037285 ┆ Christ Chapel           ┆ 2100 Cameron Drive, Unit APART… ┆ US      │
│ S1-755362802 ┆ Prabhav Business Center ┆ 797, Lake Town Block A, Kolkat… ┆ India   │
└──────────────┴─────────────────────────┴─────────────────────────────────┴─────────┘

Schema:
S

In [7]:
s1_lazy = pl.scan_csv(
    TRAIN_S1,
    separator="\t",
    infer_schema_length=10000,
)

s2_lazy = pl.scan_csv(
    TRAIN_S2,
    separator="\t",
    infer_schema_length=10000,
)

s3_lazy = pl.scan_csv(
    TRAIN_S3,
    separator="\t",
    infer_schema_length=10000,
)

print(s1_lazy.collect_schema())
print(s2_lazy.collect_schema())
print(s3_lazy.collect_schema())

Schema({'entity_id': String, 'business_name': String, 'business_address': String, 'country': String})
Schema({'entity_id': String, 'business_name': String, 'business_address': String, 'country': String})
Schema({'entity_id': String, 'business_name': String, 'business_address': String, 'country': String})


In [8]:
def get_stats(lazy_df, name):
    result = (
        lazy_df
        .select([
            pl.len().alias("rows"),
            pl.col("business_name").null_count().alias("null_names"),
            pl.col("business_address").null_count().alias("null_addresses"),
            pl.col("country").null_count().alias("null_countries"),
        ])
        .collect()
    )
    
    print(f"\n{name}")
    print(result)


get_stats(s1_lazy, "SOURCE 1")
get_stats(s2_lazy, "SOURCE 2")
get_stats(s3_lazy, "SOURCE 3")


SOURCE 1
shape: (1, 4)
┌─────────┬────────────┬────────────────┬────────────────┐
│ rows    ┆ null_names ┆ null_addresses ┆ null_countries │
│ ---     ┆ ---        ┆ ---            ┆ ---            │
│ u32     ┆ u32        ┆ u32            ┆ u32            │
╞═════════╪════════════╪════════════════╪════════════════╡
│ 2206821 ┆ 0          ┆ 0              ┆ 0              │
└─────────┴────────────┴────────────────┴────────────────┘

SOURCE 2
shape: (1, 4)
┌─────────┬────────────┬────────────────┬────────────────┐
│ rows    ┆ null_names ┆ null_addresses ┆ null_countries │
│ ---     ┆ ---        ┆ ---            ┆ ---            │
│ u32     ┆ u32        ┆ u32            ┆ u32            │
╞═════════╪════════════╪════════════════╪════════════════╡
│ 5034616 ┆ 0          ┆ 168967         ┆ 0              │
└─────────┴────────────┴────────────────┴────────────────┘

SOURCE 3
shape: (1, 4)
┌─────────┬────────────┬────────────────┬────────────────┐
│ rows    ┆ null_names ┆ null_addresses ┆ n

In [9]:
def country_stats(lazy_df, name):
    print(f"\n{name}")
    
    result = (
        lazy_df
        .group_by("country")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .collect()
    )
    
    print(result)


country_stats(s1_lazy, "SOURCE 1")
country_stats(s2_lazy, "SOURCE 2")
country_stats(s3_lazy, "SOURCE 3")


SOURCE 1
shape: (2, 2)
┌─────────┬─────────┐
│ country ┆ count   │
│ ---     ┆ ---     │
│ str     ┆ u32     │
╞═════════╪═════════╡
│ US      ┆ 1323633 │
│ India   ┆ 883188  │
└─────────┴─────────┘

SOURCE 2
shape: (2, 2)
┌─────────┬─────────┐
│ country ┆ count   │
│ ---     ┆ ---     │
│ str     ┆ u32     │
╞═════════╪═════════╡
│ US      ┆ 3016817 │
│ India   ┆ 2017799 │
└─────────┴─────────┘

SOURCE 3
shape: (2, 2)
┌─────────┬─────────┐
│ country ┆ count   │
│ ---     ┆ ---     │
│ str     ┆ u32     │
╞═════════╪═════════╡
│ US      ┆ 3170056 │
│ India   ┆ 2115547 │
└─────────┴─────────┘


In [10]:
gt_lazy = pl.scan_csv(
    GROUND_TRUTH,
    separator="\t",
    infer_schema_length=10000,
)

gt = gt_lazy.collect()

print(gt.shape)
print(gt.head())

(2206821, 2)
shape: (5, 2)
┌───────────────────┬─────────────────────────────────┐
│ source1_entity_id ┆ matched_entity_ids              │
│ ---               ┆ ---                             │
│ str               ┆ str                             │
╞═══════════════════╪═════════════════════════════════╡
│ S1-965667         ┆ S2-681193310,S2-743505751,S3-7… │
│ S1-55344266       ┆ S2-249013014,S2-197070651,S3-4… │
│ S1-343815751      ┆ S2-790675320,S2-479876582,S3-8… │
│ S1-656753428      ┆ S2-153058913,S2-24659151,S3-67… │
│ S1-102811957      ┆ S2-478959098,S2-553508714,S2-6… │
└───────────────────┴─────────────────────────────────┘


In [11]:
import unicodedata
import re

def normalize_unicode(text):
    if text is None:
        return ""
    
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.lower().strip()
    
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

In [12]:
def detect_script(text):
    if not text:
        return "empty"
    
    has_devanagari = bool(re.search(r"[\u0900-\u097F]", text))
    has_latin = bool(re.search(r"[a-zA-Z]", text))
    
    if has_devanagari and has_latin:
        return "mixed"
    elif has_devanagari:
        return "devanagari"
    elif has_latin:
        return "latin"
    else:
        return "other"

In [13]:
def detect_script(text):
    if not text:
        return "empty"
    
    has_devanagari = bool(re.search(r"[\u0900-\u097F]", text))
    has_latin = bool(re.search(r"[a-zA-Z]", text))
    
    if has_devanagari and has_latin:
        return "mixed"
    elif has_devanagari:
        return "devanagari"
    elif has_latin:
        return "latin"
    else:
        return "other"

In [14]:
from unidecode import unidecode

def transliterate(text):
    if not text:
        return ""
    
    return unidecode(text).lower().strip()

In [15]:
x = "शिव शक्ति रेस्टोरेंट"

print("Original:     ", x)
print("Transliterated:", transliterate(x))

Original:      शिव शक्ति रेस्टोरेंट
Transliterated: shiv shkti resttorentt


In [16]:
def normalize_columns(df):
    return (
        df
        .with_columns([
            pl.col("business_name")
              .fill_null("")
              .map_elements(normalize_unicode, return_dtype=pl.String)
              .alias("name_norm"),
            
            pl.col("business_address")
              .fill_null("")
              .map_elements(normalize_unicode, return_dtype=pl.String)
              .alias("address_norm"),
        ])
        .with_columns([
            pl.col("name_norm")
              .map_elements(detect_script, return_dtype=pl.String)
              .alias("name_script"),
            
            pl.col("name_norm")
              .map_elements(transliterate, return_dtype=pl.String)
              .alias("name_translit"),
            
            pl.col("address_norm")
              .map_elements(transliterate, return_dtype=pl.String)
              .alias("address_translit"),
        ])
    )

In [17]:
def normalize_lazy(lf):
    return (
        lf
        .with_columns([
            pl.col("business_name")
                .fill_null("")
                .str.normalize("NFKC")
                .str.to_lowercase()
                .str.replace_all(r"[^\p{L}\p{N}\s]", " ")
                .str.replace_all(r"\s+", " ")
                .str.strip_chars()
                .alias("name_norm"),

            pl.col("business_address")
                .fill_null("")
                .str.normalize("NFKC")
                .str.to_lowercase()
                .str.replace_all(r"[^\p{L}\p{N}\s]", " ")
                .str.replace_all(r"\s+", " ")
                .str.strip_chars()
                .alias("address_norm"),

            pl.col("country")
                .fill_null("")
                .str.normalize("NFKC")
                .str.to_lowercase()
                .str.strip_chars()
                .alias("country_norm"),
        ])
    )

In [18]:
from unidecode import unidecode

def add_transliteration(df):
    return (
        df
        .with_columns([
            pl.col("name_norm")
                .map_elements(
                    lambda x: unidecode(x) if x else "",
                    return_dtype=pl.String
                )
                .alias("name_translit"),

            pl.col("address_norm")
                .map_elements(
                    lambda x: unidecode(x) if x else "",
                    return_dtype=pl.String
                )
                .alias("address_translit"),
        ])
    )

In [19]:
def add_script_features(df):
    return (
        df
        .with_columns([
            pl.col("name_norm")
                .map_elements(
                    detect_script,
                    return_dtype=pl.String
                )
                .alias("name_script")
        ])
    )

In [20]:
def process_source(input_path, output_path):
    print(f"Processing: {input_path}")
    
    lf = pl.scan_csv(
        input_path,
        separator="\t",
        infer_schema_length=10000,
    )
    
    lf = normalize_lazy(lf)
    
    df = lf.collect()
    
    print(f"Loaded: {len(df):,} rows")
    
    df = add_transliteration(df)
    df = add_script_features(df)
    
    print("Writing:", output_path)
    
    df.write_parquet(
        output_path,
        compression="zstd",
    )
    
    print("Done.")
    
    del df
    gc.collect()

In [21]:
NORMALIZED_DIR = DATA_ROOT / "normalized"
NORMALIZED_DIR.mkdir(parents=True, exist_ok=True)

NORM_S1 = NORMALIZED_DIR / "train_source1.parquet"
NORM_S2 = NORMALIZED_DIR / "train_source2.parquet"
NORM_S3 = NORMALIZED_DIR / "train_source3.parquet"

In [22]:
process_source(TRAIN_S1, NORM_S1)

Processing: dataset\train\train_source1.tsv
Loaded: 2,206,821 rows
Writing: normalized\train_source1.parquet
Done.


In [23]:
process_source(TRAIN_S1, NORM_S1)

Processing: dataset\train\train_source1.tsv
Loaded: 2,206,821 rows
Writing: normalized\train_source1.parquet
Done.


In [24]:
process_source(TRAIN_S2, NORM_S2)

Processing: dataset\train\train_source2.tsv
Loaded: 5,034,616 rows
Writing: normalized\train_source2.parquet
Done.


In [25]:
process_source(TRAIN_S3, NORM_S3)

Processing: dataset\train\train_source3.tsv
Loaded: 5,285,603 rows
Writing: normalized\train_source3.parquet
Done.


In [28]:
s1 = pl.read_parquet(NORM_S2)

print(s1.shape)
print(s1.columns)

s1.head()

(5034616, 10)
['entity_id', 'business_name', 'business_address', 'country', 'name_norm', 'address_norm', 'country_norm', 'name_translit', 'address_translit', 'name_script']


entity_id,business_name,business_address,country,name_norm,address_norm,country_norm,name_translit,address_translit,name_script
str,str,str,str,str,str,str,str,str,str
"""S2-166376419""","""राम मार्केटिंग प्राइवेट लिमिटे…","""KH NO. -570/13, NEW DELHI, WES…","""India""","""र म म र क ट ग प र इव ट ल म ट ड""","""kh no 570 13 new delhi west de…","""india""","""r m m r k tt g p r iv tt l m t…","""kh no 570 13 new delhi west de…","""devanagari"""
"""S2-764573417""","""-- Holloway Peak Inc Seafood""","""105 ELM ST, MORGANTON, NC""","""US""","""holloway peak inc seafood""","""105 elm st morganton nc""","""us""","""holloway peak inc seafood""","""105 elm st morganton nc""","""latin"""
"""S2-639257739""","""आदित्य प्रॉपर्टीज एलएलपी""","""G-3/571, GULMOHAR COLONY, BHOP…","""India""","""आद त य प र पर ट ज एलएलप""","""g 3 571 gulmohar colony bhopal…","""india""","""aad t y p r pr tt j elelp""","""g 3 571 gulmohar colony bhopal…","""devanagari"""
"""S2-163963287""","""Summit Inc""","""GREENSBORO, NC, 19 1/2 STARDUS…","""US""","""summit inc""","""greensboro nc 19 1 2 stardust …","""us""","""summit inc""","""greensboro nc 19 1 2 stardust …","""latin"""
"""S2-49942811""","""Delta Tetlecommunication Inc""","""914 PIERPONT AVE, CLEVELAND, O…","""US""","""delta tetlecommunication inc""","""914 pierpont ave cleveland oh""","""us""","""delta tetlecommunication inc""","""914 pierpont ave cleveland oh""","""latin"""


In [29]:
hindi_sample = (
    s1
    .filter(pl.col("name_script") == "devanagari")
    .select([
        "entity_id",
        "business_name",
        "name_norm",
        "name_translit",
        "business_address",
        "address_translit",
    ])
    .head(30)
)

hindi_sample

entity_id,business_name,name_norm,name_translit,business_address,address_translit
str,str,str,str,str,str
"""S2-166376419""","""राम मार्केटिंग प्राइवेट लिमिटे…","""र म म र क ट ग प र इव ट ल म ट ड""","""r m m r k tt g p r iv tt l m t…","""KH NO. -570/13, NEW DELHI, WES…","""kh no 570 13 new delhi west de…"
"""S2-639257739""","""आदित्य प्रॉपर्टीज एलएलपी""","""आद त य प र पर ट ज एलएलप""","""aad t y p r pr tt j elelp""","""G-3/571, GULMOHAR COLONY, BHOP…","""g 3 571 gulmohar colony bhopal…"
"""S2-615967906""","""सन कंस्ट्रक्शंस प्राइवेट लिमिट…","""सन क स ट रक श स प र इव ट ल म ट…","""sn k s tt rk sh s p r iv tt l …","""43 SUKH SAMRUDDHI, Maharashtra…","""43 sukh samruddhi maharashtra …"
"""S2-566688803""","""रियल मॉडर्न फूड लिमिटेड""","""र यल म डर न फ ड ल म ट ड""","""r yl m ddr n ph dd l m tt dd""","""SHP NO. G-4, PLOT NO. 19, SECT…","""shp no g 4 plot no 19 sector 4…"
"""S2-64510343""","""अल्फा अल टेक्नोलॉजीज प्राइवेट …","""अल फ अल ट क न ल ज ज प र इव ट ल…","""al ph al tt k n l j j p r iv t…","""NO 990 C-2 GREATER KAILASH ENC…","""no 990 c 2 greater kailash enc…"
…,…,…,…,…,…
"""S2-343318003""","""गोल्ड कृष्ण प्रॉपर्टीज प्रा. ल…","""ग ल ड क ष ण प र पर ट ज प र ल""","""g l dd k ss nn p r pr tt j p r…","""#93 212, 4206/45 SUJATA SAHKAR…","""93 212 4206 45 sujata sahkari …"
"""S2-982439832""","""सेवन हॉस्पिटैलिटी प्राइवेट लिम…","""स वन ह स प ट ल ट प र इव ट ल म …","""s vn h s p tt l tt p r iv tt l…","""5A/5 KAMAYANI HOUSING SOCIETY …","""5a 5 kamayani housing society …"
"""S2-273475397""","""प्राइम भारत सर्विसेज""","""प र इम भ रत सर व स ज""","""p r im bh rt sr v s j""","""GREATER BOMBAY, EKTA MEDOWS, B…","""greater bombay ekta medows bld…"


In [30]:
(
    s1
    .group_by("name_script")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

name_script,count
str,u32
"""latin""",4568101
"""devanagari""",258863
"""other""",197091
"""mixed""",10561


In [31]:
for path, name in [
    (NORM_S1, "S1"),
    (NORM_S2, "S2"),
    (NORM_S3, "S3"),
]:
    df = pl.read_parquet(path)
    
    print(f"\n{name}")
    
    print(
        df
        .group_by("name_script")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
    )
    
    del df
    gc.collect()


S1
shape: (1, 2)
┌─────────────┬─────────┐
│ name_script ┆ count   │
│ ---         ┆ ---     │
│ str         ┆ u32     │
╞═════════════╪═════════╡
│ latin       ┆ 2206821 │
└─────────────┴─────────┘

S2
shape: (4, 2)
┌─────────────┬─────────┐
│ name_script ┆ count   │
│ ---         ┆ ---     │
│ str         ┆ u32     │
╞═════════════╪═════════╡
│ latin       ┆ 4568101 │
│ devanagari  ┆ 258863  │
│ other       ┆ 197091  │
│ mixed       ┆ 10561   │
└─────────────┴─────────┘

S3
shape: (4, 2)
┌─────────────┬─────────┐
│ name_script ┆ count   │
│ ---         ┆ ---     │
│ str         ┆ u32     │
╞═════════════╪═════════╡
│ latin       ┆ 5021733 │
│ devanagari  ┆ 138187  │
│ other       ┆ 105867  │
│ mixed       ┆ 19816   │
└─────────────┴─────────┘


In [32]:
def add_blocking_features(df):
    return (
        df
        .with_columns([
            pl.col("name_norm")
                .str.split(" ")
                .alias("name_tokens"),

            pl.col("name_translit")
                .str.split(" ")
                .alias("name_translit_tokens"),
        ])
        .with_columns([
            pl.col("name_norm")
                .str.slice(0, 4)
                .alias("name_prefix4"),

            pl.col("name_translit")
                .str.slice(0, 4)
                .alias("translit_prefix4"),
        ])
    )

In [33]:
s1 = add_blocking_features(s1)

: 

In [ ]:
s1 = s1.with_columns([
    pl.col("address_norm")
        .str.extract_all(r"\d+")
        .alias("address_numbers")
])